<a href="https://arxiv.org/abs/2106.09685" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA Implementation Overview

This notebook implements **LoRA (Low-Rank Adaptation of Large Language Models)** as described in the paper [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685). The main idea is to reduce the number of trainable parameters during fine-tuning by injecting trainable low-rank matrices into Transformer layers, while keeping the original model weights frozen.

## Key Concepts

- **Freeze Pre-trained Weights**: The original model weights are frozen and not updated during training.
- **Low-Rank Decomposition**: Instead of updating a weight matrix \( W \), we update a low-rank decomposition \( \Delta W = A \times B \), where \( A \in \mathbb{R}^{d \times r} \) and \( B \in \mathbb{R}^{r \times d} \), with \( r \ll d \).
- **No Inference Overhead**: After training, the LoRA weights are merged with the original weights for deployment.

We'll implement this on a toy neural network for demonstration purposes.

In [ ]:
# Install required packages
!pip install torch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, original_layer, rank=4, alpha=1.0):
        super(LoRALayer, self).__init__()
        self.original_layer = original_layer
        self.rank = rank
        self.alpha = alpha
        
        # Get dimensions from the original layer
        out_features, in_features = original_layer.weight.shape
        
        # Initialize low-rank matrices A and B
        self.A = nn.Parameter(torch.randn(out_features, rank))  # eq. Low-rank reparameterization: A ~ N(0, σ²)
        self.B = nn.Parameter(torch.zeros(rank, in_features))   # B initialized to zero
        
        # Scaling factor
        self.scaling = self.alpha / self.rank
    
    def forward(self, x):
        # Forward pass through original layer
        original_output = self.original_layer(x)
        
        # Add contribution from LoRA matrices
        lora_update = torch.matmul(torch.matmul(x, self.B.t()), self.A.t()) * self.scaling
        
        return original_output + lora_update
    
    def merge_weights(self):
        # Merge LoRA weights into the original layer's weight
        with torch.no_grad():
            delta_weight = torch.matmul(self.A, self.B) * self.scaling
            self.original_layer.weight += delta_weight

In [ ]:
class ToyModel(nn.Module):
    def __init__(self):
        super(ToyModel, self).__init__()
        self.layer1 = nn.Linear(16, 32)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(32, 1)
    
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

model = ToyModel()

In [ ]:
# Replace second linear layer with LoRA version
original_layer = model.layer2
lora_layer = LoRALayer(original_layer, rank=4)
model.layer2 = lora_layer

In [ ]:
# Freeze all original model weights
for param in model.parameters():
    param.requires_grad = False

# Unfreeze LoRA parameters only
for param in model.layer2.A:
    param.requires_grad = True
for param in model.layer2.B:
    param.requires_grad = True

In [ ]:
# Generate synthetic dataset
np.random.seed(0)
torch.manual_seed(0)

batch_size = 8
input_dim = 16
X = torch.randn(batch_size, input_dim)
y = torch.randn(batch_size, 1)

In [ ]:
# Define optimizer and loss function
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)
criterion = nn.MSELoss()

# Training loop
losses = []
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Final Loss: {losses[-1]:.6f}")

In [ ]:
# Merge LoRA weights into original layer for inference
model.layer2.merge_weights()

# Remove LoRA module and replace with standard layer
merged_layer = model.layer2.original_layer
model.layer2 = merged_layer

In [ ]:
# Evaluate after merging
with torch.no_grad():
    predictions = model(X)
    eval_loss = nn.MSELoss()(predictions, y)
    print(f"Evaluation Loss after merging: {eval_loss:.6f}")

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Before LoRA
original_model = ToyModel()
print(f"Original Model Trainable Parameters: {count_parameters(original_model)}")

# After LoRA
print(f"Adapted Model Trainable Parameters: {count_parameters(model)}")

# Results Summary

In this notebook, we successfully implemented the LoRA method on a small toy model. Here's what we accomplished:

- Created a simple neural network with one hidden layer.
- Replaced the output layer with a LoRA-adapted version.
- Initialized trainable low-rank matrices \( A \) and \( B \).
- Froze all original model weights and trained only the LoRA parameters.
- Merged the LoRA weights back into the original layer for inference.
- Compared the number of trainable parameters before and after applying LoRA.

## Key Takeaways

- LoRA significantly reduces the number of trainable parameters.
- It introduces no additional inference latency since weights are merged post-training.
- The technique can be applied to any linear layer in Transformer models.

This implementation demonstrates the core mechanism behind LoRA and validates its efficiency in reducing trainable parameters while preserving model performance.